# PC algorithm

Generate synthetic stroke data and run the PC algorithm on the results.

Use the PC algorithm through the gcastle package.

## Code setup

In [1]:
import pandas as pd
import os
from castle.algorithms import PC

# Add modules to path:
import sys
sys.path.append('../')  # path to dir containing modules
from modules.create_data import create_stroke_data

2026-09-09 12:44:54,366 - /home/anna/miniconda3/envs/causal_discovery_examples/lib/python3.14/site-packages/castle/backend/__init__.py[line:36] - INFO: You can use `os.environ['CASTLE_BACKEND'] = backend` to set the backend(`pytorch` or `mindspore`).
2026-09-09 12:44:54,399 - /home/anna/miniconda3/envs/causal_discovery_examples/lib/python3.14/site-packages/castle/algorithms/__init__.py[line:36] - INFO: You are using ``pytorch`` as the backend.


## Create synthetic data

Create synthetic stroke data:

In [2]:
df_synth = create_stroke_data(n=10000, seed=42)

One-hot-encode the ethnicity column.

In [3]:
df_ohe = pd.get_dummies(df_synth['ethnicity'], prefix='ethnicity').astype(int)
df_synth = pd.concat((df_synth, df_ohe), axis='columns')

Keep a list of all columns before we start removing some:

In [4]:
all_features = df_synth.columns

In [5]:
all_features

Index(['patient_id', 'year', 'age', 'ethnicity', 'male', 'afib', 'warfarin',
       'shoe_size', 'nihss', 'thrombolysis_time', 'thrombolysis',
       'mortality_prob_no_treatment', 'mortality_odds_no_treatment',
       'mortality_log_odds_no_treatment', 'mortality_prob_treated',
       'mortality_odds_treated', 'mortality_log_odds_treated',
       'odds_ratio_treated', 'log_odds_ratio_treated',
       'probability_difference', 'mortality_probability', 'died',
       'ethnicity_asian', 'ethnicity_black', 'ethnicity_white'],
      dtype='str')

Remove repeated columns in different units:

In [6]:
cols_to_drop = [
    'patient_id',
    'ethnicity',  # replaced with OHE columns
    # 'mortality_prob_no_treatment',
    'mortality_odds_no_treatment',
    'mortality_log_odds_no_treatment',
    'mortality_prob_treated',
    'mortality_odds_treated',
    'mortality_log_odds_treated',
    'odds_ratio_treated',
    'log_odds_ratio_treated',
    'probability_difference',
    # 'mortality_probability',
    'died',
]

In [7]:
all_features = [c for c in all_features if c not in cols_to_drop]

In [8]:
df_synth_short = df_synth[all_features]

In [9]:
df_synth_short.head()

,year,age,male,afib,warfarin,shoe_size,nihss,thrombolysis_time,thrombolysis,mortality_prob_no_treatment,mortality_probability,ethnicity_asian,ethnicity_black,ethnicity_white
0,2022,66,1,0,0,10,6,301.0,1,0.190,0.1446,0,0,1
1,2019,79,1,0,0,7,1,99999.0,0,0.205,0.2050,0,0,1
2,2024,88,1,0,0,9,23,99999.0,0,0.470,0.4700,0,0,1
3,2020,74,0,0,0,10,22,309.0,1,0.440,0.3718,0,0,1
4,2021,70,1,0,0,7,9,355.0,1,0.340,0.3338,0,1,0


## Run PC algorithm

Run the PC algorithm on the more limited data with the removed columns.

Set up this dictionary with a few variations for the PC algorithm. The key will be used in the file name when the results are saved, and each value is a dict of keyword arguments for the PC algorithm.

In [10]:
dict_variations = {
    'default': dict(),
    'alpha002': dict(alpha=0.02),
}

In [11]:
dict_disc = {}
for variation_name, pc_kwargs in dict_variations.items():
    pc = PC()
    pc.learn(df_synth_short)
    
    # Convert the discovered effect matrix to a dataframe in the same
    # format as the true effects. Values are 1 where there is a link
    # and 0 otherwise.
    df_effects_disc = pd.DataFrame(
        pc.causal_matrix, columns=all_features, index=all_features)
    # Store results:
    dict_disc[variation_name] = df_effects_disc.copy()

View the result for one variation:

In [12]:
dict_disc['default']

,year,age,male,afib,warfarin,shoe_size,nihss,thrombolysis_time,thrombolysis,mortality_prob_no_treatment,mortality_probability,ethnicity_asian,ethnicity_black,ethnicity_white
year,0,0,0,0,0,0,0,0,0,1,0,0,0,0
age,0,0,0,1,0,0,1,0,0,1,0,0,0,0
male,0,0,0,0,0,0,0,0,0,1,1,0,0,0
afib,0,1,0,0,1,0,0,0,0,1,0,0,0,0
warfarin,0,0,0,1,0,0,0,0,1,0,0,0,0,0
shoe_size,0,0,0,0,0,0,0,0,1,0,0,0,0,0
nihss,0,0,0,0,0,0,0,0,0,1,0,0,0,0
thrombolysis_time,0,0,0,0,0,0,0,0,0,0,1,0,0,0
thrombolysis,0,0,0,0,0,0,1,1,0,0,1,0,0,0
mortality_prob_no_treatment,0,0,0,0,0,0,0,0,0,0,1,0,0,0


Save the results:

In [13]:
path_save = '01_pc'

for variation_name, df_effects_disc in dict_disc.items():
    file_name = f'discovered_{variation_name}.csv'
    f = os.path.join(path_save, file_name)
    df_effects_disc.to_csv(f)